# 28 - Inversion de Möbius et fonction phi d'Euler

*Série Serre 100 — distillations exécutables* (EPIC #16334)

La fonction de Möbius `μ` est l'outil d'arithmétique qui transforme une somme
sur les diviseurs en une **inversion** : elle « déconvolue » une fonction
arithmétique de sa somme sur les diviseurs. Ce carnet **mesure** ses deux
identités cardinales :

- l'identité fondamentale `∑_{d|n} μ(d) = [n = 1]` ;
- la formule de la fonction phi d'Euler par inversion, `φ(n) = ∑_{d|n} μ(d)·(n/d)`.

Tout est exécuté et vérifié en Python ci-dessous. Le pendant formel (preuves
vérifiées par le noyau Lean, `ArithmeticFunction.moebius` / `Nat.totient`) est un grain de
suite de la série.


## Plan

1. La fonction de Möbius `μ` — définition et premières valeurs.
2. L'identité fondamentale `∑_{d|n} μ(d) = [n = 1]`.
3. Inversion de Möbius et la fonction phi d'Euler.
4. Exercices.


## 1. La fonction de Möbius μ

Pour `n ≥ 1`, la fonction de Möbius `μ(n)` est :

- `0` si `n` est divisible par un carré (n est « non libre de carrés ») ;
- `(-1)^k` sinon, où `k` est le nombre de facteurs premiers **distincts** de `n`.


In [1]:
def mobius(n):
    """Fonction de Möbius : 0 si n est divisible par un carré,
    sinon (-1)**k où k = nombre de facteurs premiers distincts de n."""
    d, s, m = 2, 0, n
    while d * d <= m:
        if m % (d * d) == 0:
            return 0
        if m % d == 0:
            m //= d
            s += 1
        d += 1
    if m > 1:
        s += 1
    return -1 if s % 2 else 1

# Premières valeurs (mesurées)
print("n      :", " ".join(f"{n:>2}" for n in range(1, 31)))
print("mu(n)  :", " ".join(f"{mobius(n):>2}" for n in range(1, 31)))


n      :  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30
mu(n)  :  1 -1 -1  0 -1  1 -1  0  0  1 -1  0 -1  1  1  0 -1  0 -1  0  1  1 -1  0  0  1  0  0 -1 -1


### Lecture du résultat

Les valeurs alternent `±1` sur les entiers libres de carrés et tombent à `0` dès
qu'un carré premier divise `n` : `μ(4) = μ(8) = μ(9) = 0`, `μ(12) = 0`
(`12 = 2²·3`), mais `μ(6) = 1` (`6 = 2·3`, libre de carrés, deux facteurs →
`(-1)²`). C'est cette oscillation, et les zéros sur les carrés, qui fait toute
la force de l'inversion.


## 2. L'identité fondamentale : ∑_{d|n} μ(d) = [n = 1]

C'est la propriété cardinale de `μ` : la somme de `μ` sur les diviseurs de `n`
vaut `1` si `n = 1` et `0` sinon. Elle exprime que `μ` est l'inverse, pour la
convolution sur les diviseurs, de la fonction constante `1` — le moteur de
toute l'inversion de Möbius.


In [2]:
def divisors(n):
    """Diviseurs de n, triés."""
    out = []
    for d in range(1, int(n ** 0.5) + 1):
        if n % d == 0:
            out.append(d)
            if d != n // d:
                out.append(n // d)
    return sorted(out)

# Vérification sur 1..60 : la somme vaut 1 si et seulement si n == 1.
viol = [(n, sum(mobius(d) for d in divisors(n)))
        for n in range(1, 61)
        if sum(mobius(d) for d in divisors(n)) != (1 if n == 1 else 0)]
print(f"Vérifié pour n = 1..60 : {60 - len(viol)}/60 cas conformes.")
print("Contre-exemples :", viol if viol else "aucun")


Vérifié pour n = 1..60 : 60/60 cas conformes.
Contre-exemples : aucun


### Lecture du résultat

L'identité tient sur les 60 premiers entiers sans exception. Le cas `n = 1` est
le seul à donner `1` (son seul diviseur est `1`, `μ(1) = 1`) ; pour tout
`n > 1`, les termes `±1` s'annulent exactement — c'est la **cancellation** que
l'inversion exploite.


## 3. Inversion de Möbius et la fonction phi d'Euler

On sait que `∑_{d|n} φ(d) = n` (partager `{1, …, n}` par l'ordre exact des
éléments). C'est une somme sur les diviseurs de la forme `g(n) = ∑_{d|n} f(d)`
avec `f = φ` et `g(n) = n`. L'inversion de Möbius renvoie

    f(n) = ∑_{d|n} μ(d) · g(n/d),

d'où la formule de **Euler** :

    φ(n) = ∑_{d|n} μ(d) · (n/d).


In [3]:
def phi(n):
    """Fonction phi d'Euler : nombre d'entiers 1..n premiers entre eux avec n."""
    r, m, d = n, n, 2
    while d * d <= m:
        if m % d == 0:
            while m % d == 0:
                m //= d
            r -= r // d
        d += 1
    if m > 1:
        r -= r // m
    return r

# Confrontation : phi « direct »  vs  phi « par inversion de Möbius ».
print(f"{'n':>5} {'phi(n)':>8} {'sum mu(d)*(n/d)':>16}  ")
for n in [1, 2, 6, 12, 30, 60, 84, 100, 120, 1000]:
    lhs = phi(n)
    rhs = sum(mobius(d) * (n // d) for d in divisors(n))
    print(f"{n:>5} {lhs:>8} {rhs:>16}  {'OK' if lhs == rhs else 'MISMATCH'}")


    n   phi(n)  sum mu(d)*(n/d)  
    1        1                1  OK
    2        1                1  OK
    6        2                2  OK
   12        4                4  OK
   30        8                8  OK
   60       16               16  OK
   84       24               24  OK
  100       40               40  OK
  120       32               32  OK
 1000      400              400  OK


### Lecture du résultat

Les deux colonnes coïncident pour tous les `n` testés, y compris `n = 1000`
(`φ(1000) = 400`). La formule d'Euler n'est pas une nouvelle définition : c'est
la conséquence, par inversion de Möbius, du fait élémentaire
`∑_{d|n} φ(d) = n`. C'est le prototype de l'usage de `μ` : on part d'une somme
sur les diviseurs facile à écrire, et `μ` **inverse** la convolution pour
isoler la fonction cherchée.


## 4. Exercices


In [4]:
# Exercice 1
# Ecrire `count_squarefree(n)` : le nombre d'entiers de 1 a n qui sont libres
# de carres (c.-a-d. tels que mu(k) != 0).
# Indice : sommer l'indicateur (mu(k) != 0) pour k dans 1..n.
def count_squarefree(n):
    # TODO etudiant
    return None
print("count_squarefree(100) =", count_squarefree(100))


count_squarefree(100) = None


In [5]:
# Exercice 2
# La fonction `sigma(n)` (somme des diviseurs de n) s'écrit directement depuis
# `divisors`. Mesurer sigma(n) pour n dans 1..20.
# Indice : sigma(n) = sum(divisors(n)).
def sigma(n):
    # TODO etudiant
    return None
print("sigma(12) =", sigma(12))


sigma(12) = None


In [6]:
# Exercice 3
# Montrer par mesure que l'inversion de Möbius « inverse » la convolution :
# avec f(n) = 1 pour tout n (donc g(n) = nombre de diviseurs de n), verifier
# que  sum_{d|n} mu(d) * g(n/d)  redonne f(n) = 1.
# Indice : g(n) = len(divisors(n)) ; verifier
#   sum(mobius(d) * len(divisors(n // d)) for d in divisors(n)) == 1.
def check_involution(n):
    # TODO etudiant
    return None
print("check_involution(30) =", check_involution(30))


check_involution(30) = None


## Conclusion

La fonction de Möbius est la clé de l'inversion arithmétique : une seule
propriété — `∑_{d|n} μ(d) = [n = 1]` — suffit à **déconvoluer** toute somme sur
les diviseurs. La formule de la fonction phi d'Euler en est l'application la
plus limpide : `φ(n) = ∑_{d|n} μ(d)·(n/d)`, lue ici comme la conséquence de
`∑_{d|n} φ(d) = n`. Les deux identités sont mesurées sur des dizaines
d'entiers ; leur preuve formelle, par le noyau Lean, fait l'objet d'un carnet
pendant de la série.


## Ressources

- J.-P. Serre, *A Course in Arithmetic* — ch. VI (§ 2, séries de Dirichlet : c'est
  là que les fonctions arithmétiques apparaissent dans le livre). Le volume compte
  sept chapitres et s'achève sur les formes modulaires (ch. VII) : la fonction de
  Möbius n'y a pas de chapitre propre.
- Mathlib (Lean 4) : `ArithmeticFunction.moebius` (notation `μ`, disponible sous
  `open scoped ArithmeticFunction.Moebius`), `Nat.totient`, `Nat.divisors` — le
  pendant formel de ce carnet.

Retour à la série : [README de *Serre 100*](README.md)